# Image Reconstruction Showcase
This notebook demonstrates the custom **Non-Expansive Image Denoiser**. It uses a parallel weight predictor and a custom CUDA extension for fast image aggregation.

In [ ]:
import torch
import matplotlib.pyplot as plt
from model import NonexpansiveDenoiser, get_dataloader

# 1. Initialize Model and Load Weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NonexpansiveDenoiser(radius=7, hidden_channels=16, num_layers=2)

weights_path = 'weights/denoiser_unified.pth'
state_dict = torch.load(weights_path, map_location=device, weights_only=True)

# Map old keys to new architecture for backward compatibility
key_mapping = {
    'weight_net.conv1.weight': 'weight_net.net.0.weight',
    'weight_net.conv1.bias': 'weight_net.net.0.bias',
    'weight_net.conv2.weight': 'weight_net.net.2.weight',
    'weight_net.conv2.bias': 'weight_net.net.2.bias',
}
new_state_dict = {}
for k, v in state_dict.items():
    if k in key_mapping:
        new_state_dict[key_mapping[k]] = v
    elif k not in ['inv_dy_cuda', 'inv_dx_cuda']:
        new_state_dict[k] = v

model.load_state_dict(new_state_dict, strict=False)
model = model.to(device)
model.eval()
print('Model loaded successfully!')

In [ ]:
# 2. Get Sample Data and Run Inference
dataloader = get_dataloader(batch_size=1, patch_size=256, noise_sigma=25)
noisy_img, clean_img = next(iter(dataloader))
noisy_img, clean_img = noisy_img.to(device), clean_img.to(device)

with torch.no_grad():
    denoised_img = model(noisy_img)

# Convert to numpy for plotting
def to_numpy(tensor):
    return tensor[0].cpu().numpy().transpose(1, 2, 0)

noisy_np = to_numpy(noisy_img)
denoised_np = to_numpy(denoised_img)
clean_np = to_numpy(clean_img)

In [ ]:
# 3. Plot the Results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(noisy_np)
axes[0].set_title('Corrupted Input')
axes[0].axis('off')

axes[1].imshow(denoised_np)
axes[1].set_title('Reconstructed Output')
axes[1].axis('off')

axes[2].imshow(clean_np)
axes[2].set_title('Perfect Target')
axes[2].axis('off')

plt.tight_layout()
plt.show()